# CLIP: Connecting Text and Images

In [ ]:
from urllib.request import urlopen
from PIL import Image
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as T
from IPython.display import display
from google.colab import userdata

In [ ]:
# puppy image and text
puppy_path = "https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/chapter09/images/puppy.png"
image = Image.open(urlopen(puppy_path)).convert("RGB")
caption = "a puppy playing in the snow"
# car image and text
car_path = "https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/refs/heads/main/chapter09/images/car.png"
image_two = Image.open(urlopen(car_path)).convert("RGB")
caption_two = "Adam Standke a international man of mystery with great taste driving a supercar in the Swiss Alps"
# cat image and text
cat_path = "https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/refs/heads/main/chapter09/images/cat.png"
image_three = Image.open(urlopen(cat_path)).convert("RGB")
caption_three = "An angry cat who is owned by a blonde girl who lives next door nammed Sam"

In [ ]:
image

In [ ]:
image_two

In [ ]:
image_three

In [ ]:
from transformers import CLIPTokenizerFast, CLIPProcessor, CLIPModel

In [ ]:
model_id = "openai/clip-vit-base-patch32"
access_code = userdata.get('Multi')

In [ ]:
# Load a tokenizer to preprocess the text
clip_tokenizer = CLIPTokenizerFast.from_pretrained(model_id, token=access_code)

# Load a processor to preprocess the images
clip_processor = CLIPProcessor.from_pretrained(model_id, token=access_code)

# Main model for generating text and image embeddings
model = CLIPModel.from_pretrained(model_id, token=access_code)

In [ ]:
# Tokenize the text
dog_input = clip_tokenizer(caption, return_tensors="pt")
print(dog_input)
Adam_car = clip_tokenizer(caption_two , return_tensors="pt")
print(Adam_car)
SammyCat = clip_tokenizer(caption_three, return_tensors="pt")
print(SammyCat)

In [ ]:
# Convert our input back to tokens
print(clip_tokenizer.convert_ids_to_tokens(dog_input["input_ids"][0]))
# Convert our input back to tokens
print(clip_tokenizer.convert_ids_to_tokens(Adam_car["input_ids"][0]))
# Convert our input back to tokens
clip_tokenizer.convert_ids_to_tokens(SammyCat["input_ids"][0])

In [ ]:
# Create a text embedding
text_embedding = model.get_text_features(**dog_input)
text_embedding.shape
text_embedding_two = model.get_text_features(**Adam_car)
text_embedding_two.shape
text_embedding_three = model.get_text_features(**SammyCat)
text_embedding_three.shape

In [ ]:
images= [image, image_two, image_three]

In [ ]:
# Preprocess image
processed_image = clip_processor(
    text=None, images=images, return_tensors="pt"
)["pixel_values"]

processed_image.shape

In [ ]:
# dog image
dog = processed_image[0, :, :, :]
processed_image[0, :, :, :].shape

In [ ]:
# me image
me = processed_image[1, :, :, :]
processed_image[1, :, :, :].shape

In [ ]:
# SammyCat
samCat = processed_image[2, :, :, :]
processed_image[2, :, :, :].shape

In [ ]:
# displaying processed pil images
trans = T.ToPILImage()

In [ ]:
dog_display = trans(dog)
me_display = trans(me)
sam_display = trans(samCat)

In [ ]:
# display the PIL image
display(dog_display)

In [ ]:
display(me_display)

In [ ]:
display(sam_display)

In [ ]:
# Prepare image for visualization
img = dog.permute(*torch.arange(dog.ndim - 1, -1, -1))
img = np.einsum("ijk->jik", img)

# Visualize preprocessed image
plt.imshow(img)
plt.axis("off")

In [ ]:
# Prepare image for visualization
img = me.permute(*torch.arange(me.ndim - 1, -1, -1))
img = np.einsum("ijk->jik", img)

# Visualize preprocessed image
plt.imshow(img)
plt.axis("off")

In [ ]:
# Prepare image for visualization
img = samCat.permute(*torch.arange(samCat.ndim - 1, -1, -1))
img = np.einsum("ijk->jik", img)

# Visualize preprocessed image
plt.imshow(img)
plt.axis("off")

In [ ]:
# Create the image embeddings
dog_embedding = model.get_image_features(dog.unsqueeze(0))
me_embedding = model.get_image_features(me.unsqueeze(0))
sam_embedding = model.get_image_features(samCat.unsqueeze(0))

In [ ]:
# Normalize text embeddings
text_embedding /= text_embedding.norm(dim=-1, keepdim=True) # dog
text_embedding_two /= text_embedding_two.norm(dim=-1, keepdim=True) # me
text_embedding_three /= text_embedding_three.norm(dim=-1, keepdim=True) # Sam

# Normalize image embeddings
dog_embedding /= dog_embedding.norm(dim=-1, keepdim=True) # dog
me_embedding /= me_embedding.norm(dim=-1, keepdim=True) # me
sam_embedding /= sam_embedding.norm(dim=-1, keepdim=True) # Sam

# Calculate dog similarity
text_embedd = text_embedding.detach().numpy() # dog text
dog_embedd = dog_embedding.detach().numpy() # dog image
print(np.dot(text_embedd, dog_embedd.T))

me_embedding = me_embedding.detach().numpy()
print(np.dot(text_embedd, me_embedding.T)) # dog text/me image

sam_embedding = sam_embedding.detach().numpy()
print(np.dot(text_embedd, sam_embedding.T)) # dog text/sam the cat image


# BLIP-2: Making Text Generation Models Multimodal


---



In [ ]:
from transformers import AutoProcessor, Blip2ForConditionalGeneration
import torch

# Load processor and main model
blip_processor = AutoProcessor.from_pretrained("Salesforce/blip2-opt-2.7b", token=access_code)
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16,
    token=access_code
)

# Send the model to GPU to speed up inference
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Image captioning

In [ ]:
# orginal image
me_path = "https://raw.githubusercontent.com/AdamClarkStandke/GenerativeDeepLearning/refs/heads/main/IMG_0563.jpg"
image = Image.open(urlopen(me_path)).convert("RGB")
image

In [ ]:
print(image.width)
print(image.height)

In [ ]:
# Preprocess the image to be used by the BLIP 2 model
inputs = blip_processor(image, return_tensors="pt").to(device, torch.float16)
print(inputs["pixel_values"])
print(inputs["pixel_values"].shape)

In [ ]:
# processed image that is feed into Blip 2 model for text generation
Image.fromarray(inputs["pixel_values"].squeeze(0).permute(1,2, 0).cpu().numpy(), "RGB")

In [ ]:
# Generate image ids to be passed to the decoder (LLM)
generated_ids = model.generate(**inputs, max_new_tokens=20)

# Generate text from the image ids
generated_text = blip_processor.batch_decode(
    generated_ids, skip_special_tokens=True
)
generated_text = generated_text[0].strip()
generated_text

# Visual Question Answering

In [ ]:
random_image = "https://raw.githubusercontent.com/AdamClarkStandke/TinyMachineLearning/refs/heads/main/arduinoDrone_schem.png"
image = Image.open(urlopen(random_image)).convert("RGB")
image

In [ ]:
# Visual question answering
prompt = "Question: what is this image? Answer:"
# Process both the image and the prompt
inputs = blip_processor(image, text=prompt, return_tensors="pt").to(device, torch.float16)

# Generate text
generated_ids = model.generate(**inputs, max_new_tokens=50)

# Decode generated ids into human readable text
generated_text = blip_processor.batch_decode(
    generated_ids, skip_special_tokens=True
)

generated_text = generated_text[0].strip()
generated_text

In [ ]:
# Chatting with BLIP-2 over the picture
# Visual question answering
prompt = "Question: what is this image? Answer: this is a schematic diagram of a simple circuit. Question: can it be something else? Answer:"
# Process both the image and the prompt
inputs = blip_processor(image, text=prompt, return_tensors="pt").to(device, torch.float16)

# Generate text
generated_ids = model.generate(**inputs, max_new_tokens=50)

# Decode generated ids into human readable text
generated_text = blip_processor.batch_decode(
    generated_ids, skip_special_tokens=True
)

generated_text = generated_text[0].strip()
generated_text

In [ ]:
from IPython.display import HTML, display
import ipywidgets as widgets
import re


def text_eventhandler(*args):
  random_image = "https://raw.githubusercontent.com/AdamClarkStandke/TinyMachineLearning/refs/heads/main/arduinoDrone_schem.png"
  image = Image.open(urlopen(random_image)).convert("RGB")
  question = args[0]["new"]
  pattern = r"\bAnswer:[^.]+"
  if question:
    args[0]["owner"].value = ""

    # Create prompt
    if not memory:
      prompt = "Question: " + question + "? Answer:"
    else:
      template = ""
      for i in range(len(memory)):
        template += memory[i][1][0]
      template += "Question: " + question + "? Answer:"
      prompt = template
    print(f"Prompt:{prompt}")

    # Generate text
    inputs = blip_processor(image, text=prompt, return_tensors="pt")
    inputs = inputs.to(device, torch.float16)
    generated_ids = model.generate(**inputs, max_new_tokens=50)
    generated_text = blip_processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )
    # Add period to answer
    generated_text[0] += "."

    # Update memory
    memory.append((question, generated_text))

    # print generated text
    matches = re.findall(pattern, generated_text[0])
    if matches:
      blip_text = matches[-1].strip().split("Answer:")[1]
      # Assign to output
      output.append_display_data(HTML("<b>USER:</b> " + question))
      output.append_display_data(HTML("<b>BLIP-2:</b> " + blip_text))
      output.append_display_data(HTML("<br>"))

# prepare text widget
in_text = widgets.Text()
in_text.continuous_update = False
in_text.observe(text_eventhandler, "value")
output = widgets.Output()
memory = []

# Display chat box
display(
    widgets.VBox(
        children=[output, in_text],
        layout=widgets.Layout(display="inline-flex", flex_flow="column-reverse"),
    )
)